# Notebook 01e · Milan, AlphaEarth embeddings, CLMS — all points, no train/test split

Test variant of `01_IMD_Prediction_Milan_blockCV_embedding_CLMS.ipynb` /
`01c_IMD_Prediction_Milan_blockCV_embedding_GHSL.ipynb`, requested to check what
changes when the spatial train/test split is removed. **Every one of the 3500
sample points is used both to tune the Random Forest (still via spatial-block
CV, so the tuning signal stays honest) and to fit the final model.** RF only
-- SVR/MLP are dropped to keep this focused on the one question being tested.

## What this means for evaluation

There is no held-out set here, so there is no honest in-sample holdout metric
to report -- a model scored against points it was trained on will look better
than it really is, by construction. This notebook does not compute one. The
only honest check of whether this changes anything is the fully independent
EarthLabel ground truth (450 photo-interpreted plots, never used in training
either way), scored separately after this notebook exports its raster.

## Pipeline

| Cell | Does |
|---|---|
| 1 | Config |
| 2 | AOI (from the local `data/aoi_milan/milano_aoi.gpkg`, not an EE asset -- avoids depending on any one GEE project having it uploaded), AlphaEarth embeddings, all 3500 points |
| 3 | Spatial-block helper functions (identical to 01/01c) |
| 4 | RF hyperparameter tuning, 5-fold spatial-block CV, **on all points** |
| 5 | Fit final RF on all points; train the GEE classifier on all points; export the predicted raster |

> Prerequisite: `./data/sample_points/sample_points_all_CLMS_Milan.gpkg` (already exists, from the original 01/01c run).


## Cell 1 · Imports & configuration

In [1]:
# Earth Engine project id comes from .env (copy .env.example to .env).
# Authentication is a one-off per machine: run `earthengine authenticate`.
import ee
from gee_init import init_gee

init_gee()

import json
import os
import time
import warnings

import geopandas as gpd
import joblib
import numpy as np
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload
from scipy.spatial import cKDTree
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

warnings.filterwarnings('ignore')

# -- Configuration, identical to 01/01c except as noted ------------------------
AEF_YEAR      = 2018
RANDOM_STATE  = 42
N_JOBS        = -1    # parallel -- original 01/01c use 1, but that is far too slow for an
                       # automated run; n_jobs only affects wall-clock time, not which
                       # params RandomizedSearchCV tries or their results (random_state
                       # and the cv splits are unaffected), so this is purely a speed fix
N_FOLDS       = 5                     # spatial-block CV folds for tuning
BLOCK_SIZES   = [500, 1000, 2000]     # block sizes tested for HP tuning CV
N_ITER_RF     = 40
N_CLASSES     = 7

# -- Paths -- a NEW output directory, so this never overwrites the primary
# 01/01c run that the rest of the project (validation, the paper) depends on.
TARGET        = 'CLMS'
ALL_POINTS_PATH = './data/sample_points/sample_points_all_CLMS_Milan.gpkg'
OUTPUT_DIR      = './output/milan/clms/embedding_alltrain'
os.makedirs(OUTPUT_DIR, exist_ok=True)

DRIVE_FOLDER = 'GEE_exports'

print(f'Milan, AlphaEarth, {TARGET} -- ALL POINTS, no train/test split')
print(f'points: {ALL_POINTS_PATH}')
print(f'output: {OUTPUT_DIR}')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


GEE initialised (project: imd-vietnam).


Milan, AlphaEarth, CLMS -- ALL POINTS, no train/test split
points: ./data/sample_points/sample_points_all_CLMS_Milan.gpkg
output: ./output/milan/clms/embedding_alltrain


---
## Cell 2 · AOI, AlphaEarth embeddings, all sample points

In [2]:
from shapely.geometry import mapping

# AOI from the local file, not an EE asset -- this notebook must not depend on
# any one GEE project having milano_aoi uploaded (it may not: GEE quota is
# per-project, and this project may differ from the one 01/01c were run under).
aoi_gdf  = gpd.read_file('./data/aoi_milan/milano_aoi.gpkg').to_crs('EPSG:4326')
aoi_geom = ee.Geometry(mapping(aoi_gdf.geometry.union_all()))
print(f'Milan AOI area: {aoi_geom.area(1).getInfo() / 1e6:,.0f} km2')

non_water_mask = (ee.ImageCollection('ESA/WorldCover/v200')
                  .filter(ee.Filter.bounds(aoi_geom))
                  .mosaic().clip(aoi_geom)).neq(80)

start_date = ee.Date.fromYMD(AEF_YEAR, 1, 1)
aef_col = (ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
           .filter(ee.Filter.date(start_date, start_date.advance(1, 'year')))
           .filter(ee.Filter.bounds(aoi_geom)))
aef_proj = aef_col.first().projection()
aef_img = (aef_col.mosaic()
           .setDefaultProjection(aef_proj)
           .clip(aoi_geom)
           .updateMask(non_water_mask))

band_names = aef_img.bandNames().getInfo()
print(f'AEF bands: {len(band_names)}')

# -- ALL points, no split --------------------------------------------------
gdf_all = gpd.read_file(ALL_POINTS_PATH).dropna().reset_index(drop=True)
if gdf_all['IMD_class'].dtype != int:
    raw = sorted(gdf_all['IMD_class'].unique())
    gdf_all['IMD_class'] = gdf_all['IMD_class'].map({v: i for i, v in enumerate(raw)})
gdf_all['IMD_class'] = gdf_all['IMD_class'].astype(int)

gdf_utm = gdf_all.to_crs('EPSG:32632')
coords_all = np.column_stack([gdf_utm.geometry.x, gdf_utm.geometry.y])

X_all = gdf_all[band_names].values.astype(np.float32)
y_all = gdf_all['IMD'].values.astype(np.float32)

print(f'Total points (ALL used for training): {len(gdf_all)}')
print(f'X shape: {X_all.shape} | y range: {y_all.min():.0f}-{y_all.max():.0f}%')
for c in range(N_CLASSES):
    print(f'  class {c}: {int((gdf_all["IMD_class"] == c).sum())}')

Milan AOI area: 4,455 km2


AEF bands: 64
Total points (ALL used for training): 3500
X shape: (3500, 64) | y range: 0-100%
  class 0: 500
  class 1: 500
  class 2: 500
  class 3: 500
  class 4: 500
  class 5: 500
  class 6: 500


---
## Cell 3 · Spatial block helper functions

Identical to 01/01c.

In [3]:
def assign_blocks(coords, block_size):
    bx = (coords[:, 0] // block_size).astype(int)
    by = (coords[:, 1] // block_size).astype(int)
    return np.array([f'{x}_{y}' for x, y in zip(bx, by)])

def blocks_to_folds(block_ids, n_folds, seed):
    rng    = np.random.RandomState(seed)
    unique = rng.permutation(np.unique(block_ids))
    bmap   = {b: i % n_folds for i, b in enumerate(unique)}
    return np.array([bmap[b] for b in block_ids])

def make_spatial_splits(point_folds, n_folds):
    all_idx = np.arange(len(point_folds))
    return [(all_idx[point_folds != f], all_idx[point_folds == f])
            for f in range(n_folds)]

print('Spatial block functions defined.')

Spatial block functions defined.


---
## Cell 4 · RF hyperparameter tuning, 5-fold spatial-block CV, on ALL points

In [4]:
PARAM_SPACE = {
    'n_estimators':      [100, 200, 300, 500],
    'max_depth':         [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', 0.3, 0.5],
    'max_samples':       [None, 0.7, 0.8, 0.9],
}

best_rmse_tune, best_search, best_block = np.inf, None, None
for bs in BLOCK_SIZES:
    bids = assign_blocks(coords_all, bs)
    pf = blocks_to_folds(bids, N_FOLDS, RANDOM_STATE)
    cv_splits = make_spatial_splits(pf, N_FOLDS)
    t0 = time.time()
    search = RandomizedSearchCV(
        estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
        param_distributions=PARAM_SPACE, n_iter=N_ITER_RF, cv=cv_splits,
        scoring='neg_root_mean_squared_error', n_jobs=N_JOBS,
        random_state=RANDOM_STATE, refit=True)
    search.fit(X_all, y_all)
    cv_rmse = -search.best_score_
    print(f'{bs}m spatial CV -> RMSE={cv_rmse:.3f}  ({time.time() - t0:.1f}s)')
    if cv_rmse < best_rmse_tune:
        best_rmse_tune, best_search, best_block = cv_rmse, search, bs

best_params = best_search.best_params_
print(f'\nBEST: block={best_block}m  CV_RMSE={best_rmse_tune:.3f}')
print(f'params={best_params}')

500m spatial CV -> RMSE=13.157  (286.2s)


1000m spatial CV -> RMSE=13.171  (267.9s)


2000m spatial CV -> RMSE=13.233  (262.2s)

BEST: block=500m  CV_RMSE=13.157
params={'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_samples': None, 'max_features': 0.5, 'max_depth': None}


---
## Cell 5 · Fit final RF on all points, export the predicted raster

In [5]:
rf_fitted = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, **best_params)
rf_fitted.fit(X_all, y_all)
joblib.dump(rf_fitted, f'{OUTPUT_DIR}/best_model_RF_alltrain.joblib')

# In-sample fit, printed for sanity only -- NOT a holdout accuracy claim.
# Every point here was also used to train the model, so this is optimistic
# by construction. Real evaluation is the EarthLabel comparison, run
# separately after this raster is exported and downloaded.
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
y_fit = np.clip(rf_fitted.predict(X_all), 0, 100)
print('In-sample fit (NOT a holdout metric -- these points were also used to train):')
print(f'  RMSE={np.sqrt(mean_squared_error(y_all, y_fit)):.2f}  '
      f'MAE={mean_absolute_error(y_all, y_fit):.2f}  R2={r2_score(y_all, y_fit):.3f}')

# -- GEE classifier, trained on ALL points -------------------------------------
features = [ee.Feature(None, {**{b: float(row[b]) for b in band_names}, 'IMD': float(row['IMD'])})
            for _, row in gdf_all.iterrows()]
train_fc = ee.FeatureCollection(features)

clf = (ee.Classifier.smileRandomForest(
           numberOfTrees=best_params.get('n_estimators', 200),
           minLeafPopulation=best_params.get('min_samples_leaf', 1),
           seed=RANDOM_STATE)
       .setOutputMode('REGRESSION')
       .train(features=train_fc, classProperty='IMD', inputProperties=band_names))

pred_img = (aef_img.classify(clf).rename('IMD_predicted')
            .clip(aoi_geom).updateMask(non_water_mask).clamp(0, 100))

fname = 'IMD_predicted_RF_alltrain_Milan'
out_tif = f'{OUTPUT_DIR}/{fname}.tif'
if os.path.exists(out_tif):
    print(f'raster already present at {out_tif} -- skipping export.')
else:
    task = ee.batch.Export.image.toDrive(
        image=pred_img, description=fname, folder=DRIVE_FOLDER,
        fileNamePrefix=fname, region=aoi_geom, scale=10, crs='EPSG:32632',
        maxPixels=1e10, fileFormat='GeoTIFF')
    task.start()
    print(f'submitted batch export task (id={task.id})')
    state = 'READY'
    while state not in ('COMPLETED', 'FAILED', 'CANCELLED'):
        time.sleep(15)
        state = task.status()['state']
        print(f'  {state}')
    if state != 'COMPLETED':
        raise RuntimeError(f'export {state} -- {task.status()}')

    creds = ee.data.get_persistent_credentials()
    drive = gbuild('drive', 'v3', credentials=creds)
    resp = drive.files().list(q=f"name = '{fname}.tif' and trashed = false",
                               fields='files(id, createdTime)').execute()
    files = sorted(resp.get('files', []), key=lambda f: f['createdTime'], reverse=True)
    if not files:
        raise RuntimeError(f'export completed but {fname}.tif not found on Drive.')
    request = drive.files().get_media(fileId=files[0]['id'])
    with open(out_tif, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    print(f'downloaded -> {out_tif}')

meta = {
    'target': TARGET, 'band_names': band_names, 'n_points_train': len(gdf_all),
    'best_block_m': best_block, 'cv_rmse_tune': best_rmse_tune, 'best_params': best_params,
    'in_sample_rmse': float(np.sqrt(mean_squared_error(y_all, y_fit))),
    'in_sample_mae': float(mean_absolute_error(y_all, y_fit)),
    'in_sample_r2': float(r2_score(y_all, y_fit)),
    'note': 'No train/test split -- all points used for both tuning and the final fit. '
            'in_sample_* are NOT holdout metrics. Evaluate against EarthLabel instead.',
}
with open(f'{OUTPUT_DIR}/model_metadata_alltrain.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)
print(f'\nSaved -> {OUTPUT_DIR}/model_metadata_alltrain.json')

In-sample fit (NOT a holdout metric -- these points were also used to train):
  RMSE=4.86  MAE=3.56  R2=0.981


submitted batch export task (id=VFBDEXOZDN375J4O5A74TX55)


  RUNNING


  RUNNING


  RUNNING


...

  RUNNING


  RUNNING


  RUNNING


  COMPLETED


downloaded -> ./output/milan/clms/embedding_alltrain/IMD_predicted_RF_alltrain_Milan.tif

Saved -> ./output/milan/clms/embedding_alltrain/model_metadata_alltrain.json
